## Unified Corpus
[COMETA](https://zenodo.org/records/15300719?preview_file=Fran%C3%A7ais_1049.txt) contains 17 text corpus files written in Occitan This notebook aims to unify these files into a single corpus for later tokenizer training.

In [10]:
import os
from pathlib import Path
import re
from dotenv import load_dotenv
import sys
load_dotenv()
PROJECT_ROOT = os.environ.get("PROJECT_ROOT")
sys.path.insert(0, str(Path(os.environ.get("PROJECT_ROOT", "."))))

In [ ]:

def load_medieval_occitan_corpus(data_dir: Path) -> list[str]:
    texts = []
    
    for txt_file in data_dir.rglob("*.txt"):
        with open(txt_file, "r", encoding="utf-8") as f:
            content = f.read()
        
        # Basic cleaning for medieval text
        content = re.sub(r'<[^>]+>', '', content)  # Remove XML/HTML tags
        content = re.sub(r'\[.*?\]', '', content)   # Remove editorial notes
        content = re.sub(r'\s+', ' ', content).strip()
        
        if len(content) > 100:  # Skip very short fragments
            texts.append(content)
    
    print(f"Loaded {len(texts)} texts, ~{sum(len(t) for t in texts)/1e6:.2f}M chars")
    return texts

In [8]:
input_path = Path(os.path.join(PROJECT_ROOT, "data", "raw", "COMETA_medieval_corpus"))
corpus = load_medieval_occitan_corpus(input_path)


Loaded 17 texts, ~3.86M chars


In [9]:
corpus

['Incipit planctus on am passionis dui Jhesus Christus et dolorem sue s affrinementus. Planh sobre planh dol sobre dolor Cel e terra an perdut lor senhor E yeu mon filh el solelh sa clardor Iuzieus mort a granda dezonor Ay filh tot mortal dolor Iuzieus f trop etz dezoblidatz mal vos membra del temps qu en ez passat Dels grans trebalhs dont dieu vos a gitatz de farao qu eus telva subnigatz Ay filh tan mal bon an pagat Vos le prezet de nuech come l ayro E l on menetz en las vostras mayos pueys l esta qu etz can l aguetz veyre vos En hun pilat com si fos mal fachor Ay filh tan vol es de vos Quan vos l aguietz esta cat cruzelmen Am correias l anetz batre fortmen Del cap tros pes tot cant ac fo sagnen Tota la nuech lo tenguetz el turmen ay filh e tau cruzel gen Pueyz lon poietz sobre una cadieyra E donetz li de una canavem E metiatz vous de hunga nolh en fra E diziatz li be semblas tu rey ara Ay filh la mia amor cara A pos pilatz lo menetz per iutgar Menasset li car non o volia far Car vos 

## BPE tokenizer training
1) from scratch
2) Guarantee Coverage - Esteban's vocabulary

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from pathlib import Path

# 1. Prepare your combined text corpus
corpus_files = [
    "data/garces_transcriptions.txt",      # Garces's 82M synthetic pairs + real labels
    "data/your_manuscript_ground_truth.txt", # Your 1372 manuscript transcriptions
    "data/medieval_occitan_literature.txt"   # Additional historical texts (optional)
]

# 2. Initialize a fresh tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()  # Handles spaces, special chars, unknown chars gracefully

# 3. Train on the COMBINED corpus
trainer = BpeTrainer(
    vocab_size=120,  # Slightly larger than Garces's 73 to capture common digraphs/trigraphs
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[EOS]"],
    show_progress=True
)
tokenizer.train(corpus_files, trainer)

# 4. Save
tokenizer.save("occitan_medieval_htr_tokenizer.json")
print(f"✅ Tokenizer trained with vocab size: {len(tokenizer.get_vocab())}")

In [ ]:
import json
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel

# Load Garces's tokenizer to extract his character set
with open("garces_tokenizer.json", "r", encoding="utf-8") as f:
    garces_data = json.load(f)
garces_chars = list(garces_data["model"]["vocab"].keys())

# Initialize fresh tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = ByteLevel()

# Train with Garces's characters as the initial alphabet seed
trainer = BpeTrainer(
    vocab_size=120,
    special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[EOS]"],
    initial_alphabet=garces_chars,  # ← Guarantees these characters are in the base vocab
    show_progress=True
)
tokenizer.train(corpus_files, trainer)
tokenizer.save("occitan_seeded_tokenizer.json")

## **To connect to the TrOCR**

In [ ]:
from transformers import AutoTokenizer, TrOCRProcessor, VisionEncoderDecoderModel

# Load your custom tokenizer
tokenizer = AutoTokenizer.from_pretrained("occitan_medieval_htr_tokenizer.json")

# Initialize TrOCR processor (handles image resizing + tokenizer)
processor = TrOCRProcessor(
    tokenizer=tokenizer,
    feature_extractor=None  # TrOCR handles images internally
)

# Now your decoder will predict token IDs that match your Old Occitan vocabulary
model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
    "microsoft/trocr-base-handwritten",
    "bert-base-uncased"
)
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.resize_token_embeddings(len(tokenizer))  # Align model vocab with your tokenizer

In [11]:
import json
load_dotenv()
project_root = Path(os.environ.get("PROJECT_ROOT", "."))


# Load tokenizer
with open(project_root/"data/processed/tokenizer/tokenizer_20260503_201003/occitan_tokenizer.json", "r") as f:
    data = json.load(f)


In [12]:
merges = data["model"]["merges"]  # List of ["a b", "ab c", ...]
vocab = data["model"]["vocab"]

print(f"Total merges: {len(merges)}")
print(f"Last 10 merges (least frequent):")
for merge in merges[-10:]:
    print(f"  {merge}")

# If these are rare character combos (e.g., "x̱ y̆"), you may have over-merged

Total merges: 16
Last 10 merges (least frequent):
  ['e', 'r']
  ['Ġ', 'c']
  ['o', 'n']
  ['e', 'n']
  ['a', 'n']
  ['q', 'u']
  ['Ġd', 'e']
  ['r', 'e']
  ['a', 's']
  ['Ġ', 'm']
